In [1]:
# %%
# %%
# ============================================================
# Xinhua Full Dataset Primary Topic Classification
#
# Model:
# GPT-5-mini
#
# Prompt:
# Final frozen P2-CN
#
# Task:
# Assign one dominant topic frame for each AV news article
#
# ============================================================


# %%
# ============================================================
# 0. OpenAI API setup
# ============================================================


from dotenv import load_dotenv
from openai import OpenAI

import os
import time


load_dotenv(
    "/Users/yurujia/Desktop/Dissertation Data/sentiment/.env"
)


key = os.getenv(
    "OPENAI_API_KEY"
)


if key:
    print(
        "API key loaded successfully"
    )

else:
    print(
        "API key NOT found"
    )


client = OpenAI()


MODEL = "gpt-5-mini"


print(
    "Model:",
    MODEL
)


API key loaded successfully
Model: gpt-5-mini


In [2]:


# %%
# %%
# ============================================================
# 1. Imports
# ============================================================


import pandas as pd
import json

from pathlib import Path


print(
    "Packages loaded."
)


# %%
# %%
# ============================================================
# 2. Paths
# ============================================================


BASE_DIR = Path(
    "/Users/yurujia/Desktop/Dissertation Data/China"
)


INPUT_PATH = (

    BASE_DIR /

    "descriptive_stats/"

    "xinhua_av_final_with_first_av_relevant_paragraph.xlsx"

)


BATCH_INPUT_PATH = (

    BASE_DIR /

    "xinhua_topic_P2_CN_batch_input.jsonl"

)


BATCH_OUTPUT_PATH = (

    BASE_DIR /

    "xinhua_topic_P2_CN_batch_output.jsonl"

)


OUTPUT_PATH = (

    BASE_DIR /

    "xinhua_topic_P2_CN_final.xlsx"

)


print(
    INPUT_PATH
)


Packages loaded.
/Users/yurujia/Desktop/Dissertation Data/China/descriptive_stats/xinhua_av_final_with_first_av_relevant_paragraph.xlsx


In [3]:


# %%
# %%
# ============================================================
# 3. Load dataset
# ============================================================


df = pd.read_excel(
    INPUT_PATH
)


print(
    "Dataset shape:",
    df.shape
)


print(
    df.columns.tolist()
)


display(
    df.head()
)


Dataset shape: (1155, 88)
['file', 'title', 'source', 'date', 'content', 'source_file', 'has_body_marker', 'has_wan_marker', 'has_load_date_marker', 'article_content_clean', 'extraction_method', 'extract_success', 'date_original', 'date_fixed', 'date_parsed_direct', 'year_temp', 'month_temp', 'content_char_length', 'approx_token_count', 'exclude_title_keyword', 'exclude_too_long', 'exclude_from_analysis', 'exclusion_reason', 'has_explicit_brief_title', 'has_explicit_brief_header', 'small_heading_count', 'has_multiple_small_headings', 'exclude_multi_topic_brief', 'multi_topic_brief_exclusion_reason', 'automated_screen_text', 'has_general_road_av_term', 'has_higher_automation_term', 'has_passenger_robotaxi_term', 'has_strong_passenger_robotaxi_term', 'has_bus_public_transport_term', 'has_adas_term', 'has_road_vehicle_context', 'has_strong_non_road_term', 'has_general_non_road_context', 'has_non_road_primary_topic_term', 'has_non_road_term', 'remove_no_relevant_road_av_term', 'remove_pure

,file,title,source,date,content,source_file,has_body_marker,has_wan_marker,has_load_date_marker,article_content_clean,...,av_paragraphs_checked_before_match,av_nonrelevant_substantive_paragraphs_skipped,av_skipped_nonrelevant_text,first_av_relevant_paragraph_char_count,first_av_relevant_paragraph_chinese_char_count,first_av_relevant_paragraph_english_word_count,first_av_relevant_paragraph_number_count,first_av_relevant_paragraph_approx_text_unit_count,first_paragraph_is_av_relevant,av_paragraph_same_as_original_first
0,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌.DOCX,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌,Xinhua: News in Chinese for Overseas Service,"June 2, 2017 Friday 3:43 AM GMT",（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\nXinhua: Ne...,lexis_structured1-200,True,True,True,新华社柏林６月１日电专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\n新华社记...,...,2,1,中国和德国在电动汽车领域各有优势，两国车企如果能够融合优势，将创造领先全球的强大竞争力，未来...,128,114,0,3,117,False,False
1,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东....,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东,Xinhua: News in Chinese for Overseas Service,"June 3, 2017 Saturday 4:15 AM GMT",（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,lexis_structured1-200,True,True,True,新华社柏林６月２日电专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,...,5,4,“经贸合作始终是中德关系的压舱石。李克强总理此访在经贸领域取得了丰硕的成果，必将推动中德经贸...,139,122,0,0,122,False,False
2,中国新能源汽车全球占比一半.DOCX,中国新能源汽车全球占比一半,Xinhua: News in Chinese for Overseas Service,"June 6, 2017 Tuesday 8:29 AM GMT",中国新能源汽车全球占比一半\nXinhua: News in Chinese for Ove...,lexis_structured1-200,True,True,True,新华社北京６月６日电（记者陈芳 董瑞丰）通过手机定位，找到距离最近的电动汽车，用手机开锁后就...,...,6,5,通过手机定位，找到距离最近的电动汽车，用手机开锁后就可以驾驶，直至停下来留给下一位客户使用。...,43,38,0,0,38,False,False
3,（科技）专家：自动驾驶技术有望让交通事故零伤亡.DOCX,（科技）专家：自动驾驶技术有望让交通事故零伤亡,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 7:24 AM GMT",（科技）专家：自动驾驶技术有望让交通事故零伤亡\nXinhua: News in Chine...,lexis_structured1-200,True,True,True,新华社瑞典哥德堡６月７日电（记者潘革平 付一鸣）吉利欧洲研发中心首席执行官方浩瀚日前在位于瑞...,...,1,0,NaN,79,72,0,1,73,True,True
4,（财经）日本预计于２０２３年普及５Ｇ通信.DOCX,（财经）日本预计于２０２３年普及５Ｇ通信,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 6:42 AM GMT",（财经）日本预计于２０２３年普及５Ｇ通信\nXinhua: News in Chinese ...,lexis_structured1-200,True,True,True,新华社东京６月７日电（记者钱铮）下一代超高速无线通信技术——第五代移动通信技术（５Ｇ）的商业...,...,1,0,NaN,72,60,0,2,62,True,True


In [4]:


# %%
# %%
# ============================================================
# 4. Prepare topic dataframe
# ============================================================


TEXT_COL = (
    "first_av_relevant_paragraph"
)


assert TEXT_COL in df.columns



topic_df = pd.DataFrame({

    "article_id":
        range(len(df)),


    "source":
        "Xinhua",


    "text":
        df[TEXT_COL]

})



topic_df = (
    topic_df
    .dropna(
        subset=[
            "text"
        ]
    )
)



topic_df["text"] = (

    topic_df["text"]

    .astype(str)

    .str.strip()

)



topic_df = topic_df[
    topic_df["text"].str.len() > 0
]


topic_df = (
    topic_df
    .reset_index(drop=True)
)



print(
    "Articles for topic classification:",
    len(topic_df)
)


display(
    topic_df.head()
)


Articles for topic classification: 1155


,article_id,source,text
0,0,Xinhua,在李克强总理访问德国期间，蔚来汽车５月３１日在柏林与德国大陆集团签署战略合作协议，主要涉及电...
1,1,Xinhua,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...
2,2,Xinhua,“我们正在逐渐把自动驾驶融入电动汽车这个重大专项、融入未来电动汽车产品当中。”万钢说。
3,3,Xinhua,吉利欧洲研发中心首席执行官方浩瀚日前在位于瑞典哥德堡的总部对新华社记者表示，未来汽车应用自动...
4,4,Xinhua,下一代超高速无线通信技术——第五代移动通信技术（５Ｇ）的商业使用区域预计将于２０２３年扩大至...


In [5]:



# %%
# %%
# ============================================================
# 5. Topic categories
# ============================================================


TOPICS = [

    "Technology and Innovation",

    "Safety and Risk",

    "Policy and Regulation",

    "Business and Commercialisation",

    "Public Acceptance and Trust",

    "Mobility and Social Impact",

    "Environment and Sustainability",

    "Legal and Ethics",

    "Other"

]



print(
    TOPICS
)


['Technology and Innovation', 'Safety and Risk', 'Policy and Regulation', 'Business and Commercialisation', 'Public Acceptance and Trust', 'Mobility and Social Impact', 'Environment and Sustainability', 'Legal and Ethics', 'Other']


In [6]:



# %%
# %%
# ============================================================
# 6. Final P2-CN Topic Prompt
# ============================================================


def build_prompt_p2_cn(text):

    return f"""

你是一名专门研究自动驾驶汽车新闻报道的媒体分析专家。

你的任务是识别以下自动驾驶相关新闻文本的
单一主导议题（primary frame）。

必须从下面预先定义的类别中选择且只能选择一个。

不要创建新的类别。
不要使用其他表达方式。
最终只返回英文类别名称。


可选议题：

Technology and Innovation

Safety and Risk

Policy and Regulation

Business and Commercialisation

Public Acceptance and Trust

Mobility and Social Impact

Environment and Sustainability

Legal and Ethics

Other


类别定义：


Technology and Innovation

当新闻主要关注自动驾驶技术本身的研发、改进、测试
或技术能力时选择。

包括：

- 人工智能系统
- 传感器、软件、算法
- 车辆技术系统
- 工程研发
- 技术展示
- 自动驾驶能力提升
- 技术测试和验证


Safety and Risk

当新闻主要关注自动驾驶汽车的风险、故障、事故、
可靠性或安全评估时选择。

包括：

- 交通事故
- 碰撞
- 调查
- 运行故障
- 安全担忧
- 可靠性问题


Policy and Regulation

当新闻主要关注政府行动、规则或监管框架时选择。

包括：

- 法规或立法
- 政府政策
- 监管批准
- 测试许可
- 官方标准
- 政府决定


Business and Commercialisation

当新闻主要关注经济、企业或市场活动时选择。

包括：

- 企业战略
- 投资
- 收购
- 合作
- 企业竞争
- 商业模式
- 商业化推进
- 市场发展


Public Acceptance and Trust

当新闻主要关注公众对于自动驾驶汽车的态度、
观点、信任、担忧或使用意愿时选择。

包括：

- 消费者接受度
- 公众意见
- 信任
- 担忧
- 使用意愿


Mobility and Social Impact

当新闻主要关注自动驾驶作为交通服务，
或者其对交通和社会产生的影响时选择。

包括：

- Robotaxi 服务
- 自动驾驶出租车
- 自动驾驶网约车
- 自动驾驶载客运输
- 出行便利性
- 城市交通变化


Environment and Sustainability

当新闻主要关注环境影响时选择。

包括：

- 减少排放
- 能源效率
- 可持续发展


Legal and Ethics

当新闻主要关注法律责任或伦理问题时选择。

包括：

- 法律责任
- 法律纠纷
- 伦理问题
- 问责


主要判断规则：

1.

选择最能代表新闻核心框架的议题，
而不是所有出现过的主题。


2.

不要根据孤立关键词分类。


3.

当多个主题出现时，
判断文章主要回答的问题：


技术如何发展？
→ Technology and Innovation


企业如何布局、投资或商业化？
→ Business and Commercialisation


政府如何监管？
→ Policy and Regulation


自动驾驶如何改变交通出行？
→ Mobility and Social Impact


人们是否接受和信任？
→ Public Acceptance and Trust


是否存在事故、安全风险？
→ Safety and Risk


4.

仅仅出现企业或技术，
不代表一定属于 Business 或 Technology。


5.

如果自动驾驶只是简单提及，
没有明确主题重点：

选择 Other。


最终只能返回一个英文类别名称。

不要解释。


新闻文本：

{text}

""".strip()


In [7]:



# %%
# %%
# ============================================================
# 7. Create Batch input
# ============================================================


with open(

    BATCH_INPUT_PATH,

    "w",

    encoding="utf-8"

) as f:


    for idx,row in topic_df.iterrows():


        request = {


            "custom_id":

                f"xinhua_topic_{idx}",


            "method":

                "POST",


            "url":

                "/v1/responses",


            "body":{


                "model":

                    MODEL,


                "input":

                    build_prompt_p2_cn(
                        row["text"]
                    )

            }

        }



        f.write(

            json.dumps(

                request,

                ensure_ascii=False

            )

            +

            "\n"

        )



print(
    "Batch input created:"
)


print(
    BATCH_INPUT_PATH
)


Batch input created:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_P2_CN_batch_input.jsonl


In [8]:



# %%
# %%
# ============================================================
# 8. Upload batch file
# ============================================================


batch_file = client.files.create(

    file=open(
        BATCH_INPUT_PATH,
        "rb"
    ),

    purpose="batch"

)


print(
    "Batch file ID:",
    batch_file.id
)



Batch file ID: file-Fpty5dqV8KagW4pF3TX2CH


In [9]:


# %%
# %%
# ============================================================
# 9. Create Batch Job
# ============================================================


batch_job = client.batches.create(

    input_file_id=

        batch_file.id,


    endpoint=

        "/v1/responses",


    completion_window=

        "24h"

)



print(
    "Batch ID:"
)


print(
    batch_job.id
)



# %%
# %%
# ============================================================
# 10. Wait until completed
# ============================================================


while True:


    status = client.batches.retrieve(

        batch_job.id

    )


    print(
        "Current status:",
        status.status
    )


    if status.status == "completed":

        break


    if status.status in [
        "failed",
        "expired",
        "cancelled"
    ]:

        raise Exception(
            f"Batch failed: {status.status}"
        )


    time.sleep(
        60
    )



print(
    "Batch completed."
)


Batch ID:
batch_6a66fbfdbe2c8190acb88185fc7769ed
Current status: validating
Current status: in_progress
Current status: in_progress
Current status: in_progress
Current status: in_progress
Current status: in_progress
Current status: in_progress
Current status: in_progress
Current status: finalizing
Current status: completed
Batch completed.


In [10]:



# %%
# %%
# ============================================================
# 11. Download batch output
# ============================================================


output_file_id = (

    status.output_file_id

)



file_response = client.files.content(

    output_file_id

)



with open(

    BATCH_OUTPUT_PATH,

    "w",

    encoding="utf-8"

) as f:


    f.write(
        file_response.text
    )



print(
    "Saved:"
)


print(
    BATCH_OUTPUT_PATH
)



# %%
# %%
# ============================================================
# 12. Parse results
# ============================================================


predictions = []


unexpected_outputs = []



with open(

    BATCH_OUTPUT_PATH,

    encoding="utf-8"

) as f:


    for line in f:


        item = json.loads(line)



        article_id = int(

            item["custom_id"]

            .replace(
                "xinhua_topic_",
                ""
            )

        )



        body = (

            item["response"]

            ["body"]

        )



        output_text = None



        for output_item in body["output"]:


            if output_item.get("type") == "message":


                for content_item in (

                    output_item.get(
                        "content",
                        []
                    )

                ):


                    if content_item.get("type") == "output_text":


                        output_text = (

                            content_item["text"]

                        )



        if output_text is None:

            continue



        output_text = (
            output_text
            .strip()
        )



        if output_text in TOPICS:


            predictions.append({

                "article_id":

                    article_id,


                "topic_primary_P2_CN":

                    output_text

            })


        else:


            unexpected_outputs.append({

                "article_id":

                    article_id,


                "output":

                    output_text

            })



prediction_df = pd.DataFrame(
    predictions
)



print(
    "Parsed predictions:",
    len(prediction_df)
)



print(
    "Unexpected outputs:",
    len(unexpected_outputs)
)



display(
    prediction_df.head()
)



# %%
# %%
# ============================================================
# 13. Merge predictions
# ============================================================


final_results = (

    topic_df

    .merge(

        prediction_df,

        on="article_id",

        how="left"

    )

)



final_results["model"] = MODEL


final_results["prompt"] = (
    "P2-CN"
)


final_results["classification_date"] = (

    pd.Timestamp.now()

)



print(

    "Missing topic labels:",

    final_results[
        "topic_primary_P2_CN"
    ]
    .isna()
    .sum()

)



display(
    final_results.head()
)



# %%
# %%
# ============================================================
# 14. Attach to original dataset
# ============================================================


df["article_id"] = range(
    len(df)
)



df_final = (

    df

    .merge(

        final_results[

            [

                "article_id",

                "topic_primary_P2_CN",

                "model",

                "prompt",

                "classification_date"

            ]

        ],

        on="article_id",

        how="left"

    )

)



display(
    df_final.head()
)



Saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_P2_CN_batch_output.jsonl
Parsed predictions: 1154
Unexpected outputs: 1


,article_id,topic_primary_P2_CN
0,0,Business and Commercialisation
1,1,Technology and Innovation
2,2,Technology and Innovation
3,3,Safety and Risk
4,4,Technology and Innovation


Missing topic labels: 1


,article_id,source,text,topic_primary_P2_CN,model,prompt,classification_date
0,0,Xinhua,在李克强总理访问德国期间，蔚来汽车５月３１日在柏林与德国大陆集团签署战略合作协议，主要涉及电...,Business and Commercialisation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135
1,1,Xinhua,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,Technology and Innovation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135
2,2,Xinhua,“我们正在逐渐把自动驾驶融入电动汽车这个重大专项、融入未来电动汽车产品当中。”万钢说。,Technology and Innovation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135
3,3,Xinhua,吉利欧洲研发中心首席执行官方浩瀚日前在位于瑞典哥德堡的总部对新华社记者表示，未来汽车应用自动...,Safety and Risk,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135
4,4,Xinhua,下一代超高速无线通信技术——第五代移动通信技术（５Ｇ）的商业使用区域预计将于２０２３年扩大至...,Technology and Innovation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135


,file,title,source,date,content,source_file,has_body_marker,has_wan_marker,has_load_date_marker,article_content_clean,...,first_av_relevant_paragraph_english_word_count,first_av_relevant_paragraph_number_count,first_av_relevant_paragraph_approx_text_unit_count,first_paragraph_is_av_relevant,av_paragraph_same_as_original_first,article_id,topic_primary_P2_CN,model,prompt,classification_date
0,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌.DOCX,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌,Xinhua: News in Chinese for Overseas Service,"June 2, 2017 Friday 3:43 AM GMT",（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\nXinhua: Ne...,lexis_structured1-200,True,True,True,新华社柏林６月１日电专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\n新华社记...,...,0,3,117,False,False,0,Business and Commercialisation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135
1,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东....,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东,Xinhua: News in Chinese for Overseas Service,"June 3, 2017 Saturday 4:15 AM GMT",（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,lexis_structured1-200,True,True,True,新华社柏林６月２日电专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,...,0,0,122,False,False,1,Technology and Innovation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135
2,中国新能源汽车全球占比一半.DOCX,中国新能源汽车全球占比一半,Xinhua: News in Chinese for Overseas Service,"June 6, 2017 Tuesday 8:29 AM GMT",中国新能源汽车全球占比一半\nXinhua: News in Chinese for Ove...,lexis_structured1-200,True,True,True,新华社北京６月６日电（记者陈芳 董瑞丰）通过手机定位，找到距离最近的电动汽车，用手机开锁后就...,...,0,0,38,False,False,2,Technology and Innovation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135
3,（科技）专家：自动驾驶技术有望让交通事故零伤亡.DOCX,（科技）专家：自动驾驶技术有望让交通事故零伤亡,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 7:24 AM GMT",（科技）专家：自动驾驶技术有望让交通事故零伤亡\nXinhua: News in Chine...,lexis_structured1-200,True,True,True,新华社瑞典哥德堡６月７日电（记者潘革平 付一鸣）吉利欧洲研发中心首席执行官方浩瀚日前在位于瑞...,...,0,1,73,True,True,3,Safety and Risk,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135
4,（财经）日本预计于２０２３年普及５Ｇ通信.DOCX,（财经）日本预计于２０２３年普及５Ｇ通信,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 6:42 AM GMT",（财经）日本预计于２０２３年普及５Ｇ通信\nXinhua: News in Chinese ...,lexis_structured1-200,True,True,True,新华社东京６月７日电（记者钱铮）下一代超高速无线通信技术——第五代移动通信技术（５Ｇ）的商业...,...,0,2,62,True,True,4,Technology and Innovation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124135


In [11]:


# %%
# %%
# ============================================================
# 15. Topic distribution
# ============================================================


topic_distribution = (

    df_final[

        "topic_primary_P2_CN"

    ]

    .value_counts()

)



print(
    "Topic distribution:"
)


display(
    topic_distribution
)



topic_percentage = (

    topic_distribution

    /

    topic_distribution.sum()

    *

    100

)



print(
    "Topic percentage:"
)


display(
    topic_percentage.round(2)
)



# %%
# %%
# ============================================================
# 16. Save final dataset
# ============================================================


df_final.to_excel(

    OUTPUT_PATH,

    index=False

)



print(
    "Final saved:"
)


print(
    OUTPUT_PATH
)



# %%
# %%
# ============================================================
# 17. Save unexpected outputs if any
# ============================================================


if len(unexpected_outputs) > 0:


    unexpected_path = (

        BASE_DIR /

        "xinhua_topic_P2_CN_unexpected_outputs.xlsx"

    )


    pd.DataFrame(
        unexpected_outputs
    ).to_excel(

        unexpected_path,

        index=False

    )


    print(
        "Unexpected outputs saved:"
    )


    print(
        unexpected_path
    )



# %%
# %%
# ============================================================
# Final summary
# ============================================================


print(
    "=" * 80
)


print(
    "Xinhua Topic Classification Completed"
)


print(
    "Articles:",
    len(df_final)
)


print(
    "Missing labels:",
    df_final[
        "topic_primary_P2_CN"
    ]
    .isna()
    .sum()
)


print(
    "Output:"
)


print(
    OUTPUT_PATH
)


Topic distribution:


topic_primary_P2_CN
Technology and Innovation         586
Business and Commercialisation    251
Policy and Regulation             138
Mobility and Social Impact         98
Other                              39
Safety and Risk                    16
Environment and Sustainability     11
Public Acceptance and Trust         9
Legal and Ethics                    6
Name: count, dtype: int64

Topic percentage:


topic_primary_P2_CN
Technology and Innovation         50.78
Business and Commercialisation    21.75
Policy and Regulation             11.96
Mobility and Social Impact         8.49
Other                              3.38
Safety and Risk                    1.39
Environment and Sustainability     0.95
Public Acceptance and Trust        0.78
Legal and Ethics                   0.52
Name: count, dtype: float64

Final saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_P2_CN_final.xlsx
Unexpected outputs saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_P2_CN_unexpected_outputs.xlsx
Xinhua Topic Classification Completed
Articles: 1155
Missing labels: 1
Output:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_P2_CN_final.xlsx


In [12]:
# %%
# ============================================================
# Reload manually corrected final topic dataset
# ============================================================


FINAL_PATH = (

    BASE_DIR /

    "xinhua_topic_P2_CN_final.xlsx"

)


df_final = pd.read_excel(
    FINAL_PATH
)


print(
    "Reloaded dataset:",
    df_final.shape
)


print(
    "Missing topic labels:",
    df_final[
        "topic_primary_P2_CN"
    ]
    .isna()
    .sum()
)


display(
    df_final.head()
)

Reloaded dataset: (1155, 93)
Missing topic labels: 0


,file,title,source,date,content,source_file,has_body_marker,has_wan_marker,has_load_date_marker,article_content_clean,...,first_av_relevant_paragraph_english_word_count,first_av_relevant_paragraph_number_count,first_av_relevant_paragraph_approx_text_unit_count,first_paragraph_is_av_relevant,av_paragraph_same_as_original_first,article_id,topic_primary_P2_CN,model,prompt,classification_date
0,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌.DOCX,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌,Xinhua: News in Chinese for Overseas Service,"June 2, 2017 Friday 3:43 AM GMT",（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\nXinhua: Ne...,lexis_structured1-200,True,True,True,新华社柏林６月１日电专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\n新华社记...,...,0,3,117,False,False,0,Business and Commercialisation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124
1,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东....,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东,Xinhua: News in Chinese for Overseas Service,"June 3, 2017 Saturday 4:15 AM GMT",（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,lexis_structured1-200,True,True,True,新华社柏林６月２日电专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,...,0,0,122,False,False,1,Technology and Innovation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124
2,中国新能源汽车全球占比一半.DOCX,中国新能源汽车全球占比一半,Xinhua: News in Chinese for Overseas Service,"June 6, 2017 Tuesday 8:29 AM GMT",中国新能源汽车全球占比一半\nXinhua: News in Chinese for Ove...,lexis_structured1-200,True,True,True,新华社北京６月６日电（记者陈芳 董瑞丰）通过手机定位，找到距离最近的电动汽车，用手机开锁后就...,...,0,0,38,False,False,2,Technology and Innovation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124
3,（科技）专家：自动驾驶技术有望让交通事故零伤亡.DOCX,（科技）专家：自动驾驶技术有望让交通事故零伤亡,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 7:24 AM GMT",（科技）专家：自动驾驶技术有望让交通事故零伤亡\nXinhua: News in Chine...,lexis_structured1-200,True,True,True,新华社瑞典哥德堡６月７日电（记者潘革平 付一鸣）吉利欧洲研发中心首席执行官方浩瀚日前在位于瑞...,...,0,1,73,True,True,3,Safety and Risk,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124
4,（财经）日本预计于２０２３年普及５Ｇ通信.DOCX,（财经）日本预计于２０２３年普及５Ｇ通信,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 6:42 AM GMT",（财经）日本预计于２０２３年普及５Ｇ通信\nXinhua: News in Chinese ...,lexis_structured1-200,True,True,True,新华社东京６月７日电（记者钱铮）下一代超高速无线通信技术——第五代移动通信技术（５Ｇ）的商业...,...,0,2,62,True,True,4,Technology and Innovation,gpt-5-mini,P2-CN,2026-07-27 14:53:18.124


In [13]:
# %%
# ============================================================
# Topic distribution
# ============================================================


topic_distribution = (

    df_final[
        "topic_primary_P2_CN"
    ]

    .value_counts()

)


print(
    "Topic distribution:"
)


display(
    topic_distribution
)



topic_percentage = (

    topic_distribution

    /

    topic_distribution.sum()

    *

    100

)


print(
    "Topic percentage:"
)


display(
    topic_percentage.round(2)
)

Topic distribution:


topic_primary_P2_CN
Technology and Innovation         586
Business and Commercialisation    252
Policy and Regulation             138
Mobility and Social Impact         98
Other                              39
Safety and Risk                    16
Environment and Sustainability     11
Public Acceptance and Trust         9
Legal and Ethics                    6
Name: count, dtype: int64

Topic percentage:


topic_primary_P2_CN
Technology and Innovation         50.74
Business and Commercialisation    21.82
Policy and Regulation             11.95
Mobility and Social Impact         8.48
Other                              3.38
Safety and Risk                    1.39
Environment and Sustainability     0.95
Public Acceptance and Trust        0.78
Legal and Ethics                   0.52
Name: count, dtype: float64

In [14]:
TOPIC_DIST_OUTPUT = (

    BASE_DIR /

    "xinhua_topic_distribution_P2_CN.xlsx"

)


topic_summary = pd.DataFrame({

    "count":
        topic_distribution,

    "percentage":
        topic_percentage

})


topic_summary = (
    topic_summary
    .reset_index()
)


topic_summary.columns = [
    "topic",
    "count",
    "percentage"
]


topic_summary.to_excel(
    TOPIC_DIST_OUTPUT,
    index=False
)


print(
    "Saved topic distribution:"
)


print(
    TOPIC_DIST_OUTPUT
)

Saved topic distribution:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_distribution_P2_CN.xlsx
